**Поиск аномалий (Anomaly Detection) для прогнозирования поломок оборудования**  
— это одна из самых дорогих и востребованных задач в промышленности (направлении  
 _Predictive Maintenance_ или прогнозного обслуживания).

Представьте турбину на электростанции или станок на заводе. На них навешаны сотни  
датчиков: температура, вибрация, давление, шум. Оборудование 99% времени работает  
нормально. Ломается оно редко, и каждая поломка уникальна.

Поскольку примеров поломок (разметки) почти нет, классическое обучение с учителем здесь бессильно.  
Мы используем **обучение без учителя**: обучаем модель _только на данных нормальной работы_.  
 Модель запоминает, как станок ведет себя, когда он «здоров». Как только показатели датчиков начинают  
 странно отклоняться от этой нормы, модель бьет тревогу: «Внимание, зафиксирована аномалия, скоро будет поломка!».

---


Топ-3 алгоритма для поиска аномалий без учителя

1. **Isolation Forest (Изолирующий лес)** — абсолютный хит. Он не ищет норму, он пытается _изолировать_  
   (отделить) каждую точку от остальных с помощью случайных разрезов. Аномалии висят на краях, поэтому  
   их отрезать проще и быстрее всего. Если точка изолируется за 2-3 шага — это аномалия.

2. **One-Class SVM (Одноклассовый метод опорных векторов)** — строит жесткую границу вокруг «облака»  
   нормальных данных. Всё, что вылетает за эту границу, считается аномалией.

3. **Автоэнкодеры (Нейросети)** — сжимают данные нормальной работы в узкое «горлышко» (как PCA) и  
   восстанавливают обратно. Нейросеть учится идеально восстанавливать нормальные сигналы. Если станок  
   начинает барахлить, нейросеть не сможет восстановить этот странный сигнал и выдаст высокую ошибку  
   восстановления (Reconstruction Error).


# Пример пошагово Isolation Forest

У нас есть два признака: **Вибрация** (от 0 до 5) и **Температура** (от 20 до 120).

Допустим, на заводе стоят 5 станков. Четыре из них работают нормально, а один — сломался (это наша аномалия):

- Станок А (норма): Вибрация `1.5`, Температура `40`
- Станок Б (норма): Вибрация `2.0`, Температура `45`
- Станок В (норма): Вибрация `1.8`, Температура `42`
- Станок Г (норма): Вибрация `2.1`, Температура `47`
- **Станок Х (АНОМАЛИЯ):** Вибрация `4.8`, Температура `115` (он дико трясётся и перегрелся).

Вот как алгоритм строит **одно случайное дерево**, чтобы поймать аномалию.

---

Шаг 1. Выбор случайного признака

Алгоритм бросает виртуальный кубик. Выпадает признак: **Вибрация**.  
_Робот понятия не имеет, какой станок сломан, он действует вслепую._

Шаг 2. Случайный разрез (Случайный забор)

Алгоритм смотрит на диапазон вибрации (от самого маленького `1.5` до самого большого `4.8`). И наобум выбирает число для разделения. Например, число **`3.5`**.

Он ставит первый забор: **«Вибрация > 3.5?»**

- **Группа «НЕТ»:** Станки А, Б, В, Г (у них вибрация маленькая).
- **Группа «ДА»:** **Станок Х** (у него вибрация 4.8).

**Посмотрите, что произошло!** Станок Х оказался в группе **совершенно один**. Он полностью изолирован от остальных всего за **1 шаг** (глубина = 1). Алгоритм ставит на Станок Х галочку: _«Подозрительно быстро отделился»_.

---

Шаг 3. Продолжаем делить оставшихся

Но алгоритму нужно изолировать вообще каждый станок. Он берет оставшуюся группу (А, Б, В, Г) и снова бросает кубик.

Выпадает признак: **Температура**.  
Диапазон оставшихся станков по температуре — от 40 до 47 градусов.  
Алгоритм наобум выбирает число для разреза, например, **`43.5`**.

Ставит второй забор: **«Температура > 43.5?»**

- **Группа «НЕТ»:** Станки А, В (у них температура 40 и 42).
- **Группа «ДА»:** Станки Б, Г (у них температура 45 and 47).

Никто не остался один. Все еще сидят кучками.

Шаг 4. Делим дальше

Берем группу (А, В). Снова бросаем кубик. Выпадает **Вибрация**.  
Случайный разрез: **`1.6`**.

- **Группа «НЕТ»:** Станок А (вибрация 1.5). **Ура, Станок А изолирован!** Для этого потребовалось **3 шага**.
- **Группа «ДА»:** Станок В (вибрация 1.8). **Станок В тоже изолирован!** Потребовалось **3 шага**.

_Затем алгоритм точно так же за пару шагов разделит станки Б и Г._

---

Шаг 5. Сбор статистики со ста деревьев

Алгоритм проделал это один раз. Но так как разрезы выбираются случайно, в другом дереве он мог сначала выбрать температуру и поставить забор на уровне `30` градусов (тогда бы вообще никто не отделился).

Поэтому Isolation Forest строит **100 или 200 таких случайных деревьев** и для каждого станка записывает, за сколько шагов он остался один.

В итоге получается такая ведомость:

| Станок                  | Дерево 1 | Дерево 2 | Дерево 3 | ... | **Среднее число шагов (Глубина)**  |
| ----------------------- | -------- | -------- | -------- | --- | ---------------------------------- |
| **Станок Х (Аномалия)** | 1 шаг    | 2 шага   | 1 шаг    | ... | **1.3 шага** (Отлетает мгновенно)  |
| **Станок А (Норма)**    | 3 шага   | 5 шагов  | 4 шага   | ... | **4.2 шага** (Пришлось повозиться) |
| **Станок Б (Норма)**    | 4 шага   | 3 шага   | 5 шагов  | ... | **4.0 шагов** (Сидит в толпе)      |

Финал: Вычисление «Индекса аномальности»

Алгоритм переводит эти шаги в оценку от 0 до 1:

- Если объект в среднем изолируется **очень быстро** (мало шагов) — его индекс близок к **1** (Это аномалия!).
- Если объекту нужно **много шагов**, чтобы отделиться от толпы — его индекс близок к **0** (Это норма).

В нашем примере **Станок Х** получит оценку, например, `0.85`, и система тут же забьет тревогу.

---


# Затяжная аномалия: скользящее окно

- df['разброс_3мин'] = df['температура'].rolling(window=3).std()
- df['скорость_роста'] = df['температура'].diff(periods=3)

Представьте, что станок начинает медленно умирать.

- В 10:00 температура `40`° (норма)
- В 10:01 температура `41`°
- В 10:02 температура `42`°
- ...каждую минуту она растет строго на 1 градус.
- В 11:00 температура стала `100`° (это уже опасный перегрев!).

**Почему Isolation Forest без окна её не заметит?**  
Алгоритм оценивает каждую минуту **изолированно**. Когда он будет делать случайный разрез, для него минута с температурой `60`° или `70`° покажется абсолютно нормальной, ведь в истории завода температура часто бывает такой (например, при обычной повышенной нагрузке). Алгоритм не увидит здесь ничего странного. Он не понимает, что температура _непрерывно растет уже час_.

**Как спасает окно (например, на 10 минут):**  
Окно считает **скорость изменения**.

- При нормальной работе за 10 минут температура колеблется: +1°, -1°, +0.5° (скорость близка к нулю).
- Во время поломки окно видит: за последние 10 минут температура выросла ровно на 10 градусов.  
   Для Isolation Forest этот показатель (скорость = +10) станет уникальным. Он мгновенно изолирует эти минуты, потому что в нормальной истории такого стабильного роста никогда не было.

---


| Минута  | Температура | Скорость роста (разница за 10 мин) | Разброс (стандартное отклонение) |
| ------- | ----------- | ---------------------------------- | -------------------------------- |
| ...     | ...         | ...                                | ...                              |
| 100     | 45          | **0.2** (нормальные колебания)     | **1.1** (датчик "живой")         |
| **300** | 70          | **10.0** (температура прёт вверх!) | **0.0** (датчик завис)           |


## Автоэнкодер (Autoencoder)

Если Isolation Forest ищет аномалии с помощью случайных «заборов», то Автоэнкодер делает это через «сжатие и восстановление» данных.

ЧАСТЬ 1: Как работает Автоэнкодер по шагам

Автоэнкодер — это нейросеть в форме **песочных часов**. Она состоит из двух частей:

1. **Энкодер (Сжиматель):** принимает ваши 50 признаков и сжимает их в крошечное «узкое горлышко» (например, всего в 2–3 числа).
2. **Декодер (Восстановитель):** берет эти 3 числа из горлышка и пытается развернуть их обратно в исходные 50 признаков.

Сеть обучают **только на нормальных данных** (когда станок работает исправно). Она учится идеально сжимать и восстанавливать нормальное поведение.

Шаг 1. Перепись нормальной жизни (Обучение)

Подаем на вход нормальную минуту работы: Вибрация `1.5`, Температура `40`.

- **Энкодер** сжимает эту пару в код: `[0.12]`.
- **Декодер** берет `[0.12]` и восстанавливает на выходе: Вибрация `1.51`, Температура `39.9`.
- **Результат:** Сеть восстановила данные почти идеально. Ошибка восстановления (Loss) мизерная. Сеть понимает этот режим работы.

Шаг 2. Подача аномалии в готовую сеть (Тестирование)

Станок сломался. В сеть прилетает аномальная минута: Вибрация `4.8`, Температура `115`.

- **Энкодер** никогда раньше не видел таких цифр! Но у него нет выбора, он сжимает их по своим старым правилам для нормальных данных в код, например: `[0.35]`.
- **Декодер** берет код `[0.35]` и пытается восстановить из него то, чему его учили — _нормальный станок_. На выходе он выдает: Вибрация `2.0`, Температура `45`.

Шаг 3. Подсчет Ошибки Восстановления (Reconstruction Error)

Алгоритм сравнивает то, что **пришло на вход**, с тем, что **выдала сеть на выходе**:

- На входе вибрация была `4.8`, на выходе стала `2.0`. Разница = **2.8**
- На входе температура была `115`, на выходе стала `45`. Разница = **70**

Ошибка восстановления просто колоссальная! Автоэнкодер как бы говорит: _«Я попытался нарисовать этот объект по своим правилам для нормальных данных, но у меня получилась полная чушь. Значит, этот объект — дикая аномалия!»_.

---

ЧАСТЬ 2: В чем разница с Isolation Forest и что лучше?

Нельзя сказать, что один алгоритм строго «лучше» другого. Они подходят для совершенно разных условий.

| Критерий          | Isolation Forest                                                         | Автоэнкодер (Нейросеть)                                                   |
| ----------------- | ------------------------------------------------------------------------ | ------------------------------------------------------------------------- |
| **Что ищет?**     | Ищет **редкие, одинокие** точки данных, которые легко отделить.          | Ищет данные, которые **не похожи на структуру нормальных**.               |
| **Нормализация**  | **Вообще не нужна** (дереву плевать на масштаб).                         | **Жизненно необходима** (иначе сеть ослепнет, как мы разбирали в начале). |
| **Сложные связи** | Плохо видит сложные, скрытые нелинейные зависимости между 50+ датчиками. | **Идеально видит** тончайшие скрытые зависимости между сотнями датчиков.  |
| **Объем данных**  | Отлично работает даже на маленьких таблицах (от 100 строк).              | Требует **огромного количества** данных для долгого обучения.             |
| **Железо**        | Работает на обычном процессоре ноутбука за доли секунды.                 | Требует видеокарту (GPU) и много времени на обучение.                     |

---

Что же выбрать для вашей задачи?

1. **Выбирайте Isolation Forest, если:**
   - У вас обычные табличные данные (до 50-100 признаков).
   - Данных не терабайты, а вычисления нужно делать быстро и без лишней настройки.
   - Вы не хотите возиться с нормализацией и подбором сложной архитектуры нейросети.
   - _Это идеальный базовый вариант (Baseline), с которого всегда нужно начинать._
2. **Выбирайте Автоэнкодер, если:**
   - У вас сложнейший комплекс из сотен датчиков (например, турбина самолета или огромный химзавод), где важна не просто цифра одного датчика, а их хитрая синергия.
   - У вас есть терабайты чистых, гарантированно нормальных данных для долгого обучения сети.
   - Обычные деревья не справляются и пропускают сложные, скрытые во времени аномалии.

---
